# Qwen2.5-VL GRPO: Object Counting with Temporal Consistency

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!

<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

**Goal**: Train Qwen2.5-VL-7B with GRPO (Group Relative Policy Optimization) to improve temporal consistency in object counting across video frames.

**Main Challenge**: When objects temporarily disappear and re-enter the frame, the model should count them as the SAME object, not as new instances.

## Setup Instructions for Google Colab

**Before running this notebook:**

1. Upload your dataset to Google Drive:
   - `ground_truth.csv` (1000 labeled videos)
   - `vlm_videos/` folder (1000 .mp4 files)

2. Mount Google Drive (cell below will handle this)

3. Update the paths in the Configuration cell to match your Google Drive structure

## Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass  # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2
# Fix Pillow version compatibility - Unsloth has issues with Pillow 11.x
!pip install -q "pillow<11.0.0"

## Mount Google Drive (for Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

**IMPORTANT**: Update these paths to match your Google Drive structure!

In [ ]:
from pathlib import Path
import json, re, random
import pandas as pd
from typing import List, Dict

# ---- UPDATE THESE PATHS ----
# Example: If you uploaded to "My Drive/vlm-object-counting/data/inputs/"
DRIVE_BASE = Path("/content/drive/MyDrive/vlm-object-counting/data/inputs")
CSV_PATH = DRIVE_BASE / "ground_truth.csv"
VIDEO_ROOT = DRIVE_BASE / "vlm_videos"

# ---- Video sampling config (matching supervised finetuning) ----
NUM_FRAMES_PER_VIDEO = 25  # Sample 25 frames per video
SAMPLING_STRATEGY = "uniform"  # Uniform sampling across video

# ---- Output locations ----
OUT_DIR = Path("./outputs_grpo_object_counting")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Train/Test Split ----
TRAIN_RATIO = 0.75  # 75% for training (750 samples)
TEST_RATIO = 0.25   # 25% for testing (250 samples)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)

print(f"CSV path: {CSV_PATH}")
print(f"Video root: {VIDEO_ROOT}")
print(f"CSV exists: {CSV_PATH.exists()}")
print(f"Video dir exists: {VIDEO_ROOT.exists()}")
print(f"Frames per video: {NUM_FRAMES_PER_VIDEO}")

## Load Model

In [ ]:
from unsloth import FastVisionModel
import torch

max_seq_length = 16384  # Must be this long for VLMs
lora_rank = 16  # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-VL-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # False for LoRA 16bit
    fast_inference = True,  # Enable vLLM fast inference
    gpu_memory_utilization = 0.8,  # Reduce if out of memory
)

## Configure LoRA

Note: vLLM does not yet support LoRA on vision layers, so we only add them on language layers.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # False if not finetuning vision layers
    finetune_language_layers   = True,   # False if not finetuning language layers
    finetune_attention_modules = True,   # False if not finetuning attention layers
    finetune_mlp_modules       = True,   # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    use_gradient_checkpointing = "unsloth",  # Reduces memory usage
)

## Data Preparation

### 1) Load and Parse CSV

The CSV has a special format where JSON spans multiple lines with code fences.

In [ ]:
# Read the CSV
df = pd.read_csv(CSV_PATH, header=None, names=["video_name", "ground_truth_raw"])
print(f"Total rows in CSV: {len(df)}")
df.head(3)

In [ ]:
def strip_fences(s: str) -> str:
    """Remove markdown code fences from JSON strings"""
    if not isinstance(s, str):
        return s
    s2 = s.strip()
    if s2.startswith("```"):
        s2 = re.sub(r"^```(?:json)?\s*|\s*```$", "", s2, flags=re.DOTALL).strip()
    return s2

def parse_gt_json(s: str) -> dict | None:
    """Parse ground truth JSON from CSV"""
    try:
        return json.loads(strip_fences(s))
    except Exception:
        return None

# Parse and validate
parsed = df["ground_truth_raw"].apply(parse_gt_json)
ok = parsed.apply(lambda x: isinstance(x, dict) and "object_counts" in x and isinstance(x["object_counts"], dict))
print(f"Valid rows: {ok.sum()} / {len(df)}")

df_ok = df[ok].copy().reset_index(drop=True)
df_bad = df[~ok].copy()

if not df_bad.empty:
    print(f"WARNING: {len(df_bad)} rows are invalid and will be skipped")

print(f"\nUsable dataset size: {len(df_ok)}")

### 2) Train/Test Split (750/250)

In [ ]:
# Deterministic split
idx = list(range(len(df_ok)))
random.shuffle(idx)

train_n = int(len(idx) * TRAIN_RATIO)
test_n = len(idx) - train_n

train_idx = idx[:train_n]
test_idx = idx[train_n:]

train_df = df_ok.iloc[train_idx].reset_index(drop=True)
test_df = df_ok.iloc[test_idx].reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
train_df.head(3)

### 3) Define Prompt Template with Temporal Consistency

**Key addition**: Explicit instructions about temporal consistency to address the main challenge.

In [ ]:
REASONING_START = "<REASONING>"
REASONING_END = "</REASONING>"
SOLUTION_START = "<SOLUTION>"
SOLUTION_END = "</SOLUTION>"

REQUIRED_KEYS = [
    "forklift_count",
    "pedestrian_count",
    "forklift_driver_count",
    "truck_count",
]

# Comprehensive prompt for object counting with temporal consistency
PROMPT_TEMPLATE = f"""You are an AI Industrial Safety Analyst specialized in object counting in factory video sequences.

**YOUR TASK:**
Analyze all provided video frames in sequence and count unique objects across the ENTIRE video duration.

**CRITICAL: TEMPORAL CONSISTENCY**
This is the most important aspect of your task:
- You will see multiple frames from the same video (25 frames sampled uniformly)
- Track each unique object across ALL frames
- If an object appears in frame 1, disappears in frames 2-10, and reappears in frame 11, count it as ONE object, not two
- Do NOT double-count objects that temporarily go off-screen, behind obstacles, or out of view
- Focus on unique object instances across the entire video sequence, not per-frame counts

**OBJECT DEFINITIONS:**

1. **Forklift (forklift_count):**
   - Motorized industrial lift trucks with forks
   - Include both moving AND stationary/parked forklifts
   - Include ride-on and stand-on forklifts
   - EXCLUDE: Manual pallet jacks, simple hand-operated equipment

2. **Pedestrian (pedestrian_count):**
   - Any person on foot within the scene
   - Include people operating manual pallet jacks
   - EXCLUDE: People counted as forklift drivers (see below)

3. **Forklift Driver (forklift_driver_count):**
   - Person visibly in the operator's seat/platform of a forklift
   - Must be actively controlling or about to operate the forklift
   - Do NOT count if person is just standing near a forklift
   - Driver count should be ≤ forklift count (logical consistency)

4. **Truck (truck_count):**
   - Large commercial vehicles: semi-trucks, box trucks, flatbed trucks
   - Vehicles designed for transporting goods

**OUTPUT FORMAT (MANDATORY):**
You MUST provide exactly two blocks in your response:

{REASONING_START}
Explain your counting methodology. Specifically mention:
- How you tracked objects across the 25 frames
- Any temporal consistency challenges you handled
- Your approach to avoid double-counting
(Keep to 3-4 sentences maximum)
{REASONING_END}

{SOLUTION_START}
{{
  "object_counts": {{
    "forklift_count": <integer>,
    "pedestrian_count": <integer>,
    "forklift_driver_count": <integer>,
    "truck_count": <integer>
  }}
}}
{SOLUTION_END}

**STRICT RULES:**
1. The JSON must be valid with EXACTLY those four keys
2. All values must be non-negative integers
3. NO extra keys, NO comments, NO text in the {SOLUTION_START}...{SOLUTION_END} block
4. forklift_driver_count must be ≤ forklift_count
5. If you're uncertain, provide your best estimate based on visible evidence
6. Remember: Track unique objects across the ENTIRE video sequence

Begin your analysis now."""

print(f"Prompt template created: {len(PROMPT_TEMPLATE)} characters")
print("\nFirst 400 characters:")
print(PROMPT_TEMPLATE[:400] + "...")

### 4) Create Conversation Format for RL Training

In [ ]:
from PIL import Image
import cv2

def sample_frames_from_video(video_path: Path, k: int = NUM_FRAMES_PER_VIDEO, strategy: str = SAMPLING_STRATEGY):
    """
    Sample frames from video - matches the supervised finetuning approach.
    
    Args:
        video_path: Path to the video file
        k: Number of frames to sample (default: 25)
        strategy: Sampling strategy - "uniform", "random", or "index"
    
    Returns:
        List of PIL Image frames
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count == 0:
        raise ValueError(f"{video_path} has no frames")
    
    if k <= 0:
        raise ValueError(f"Number of frames (k) must be positive, got {k}")
    if k > frame_count:
        k = frame_count
    
    # Determine frame indices based on strategy
    if strategy == "uniform":
        step = frame_count / k
        indices = [int(i * step) for i in range(k)]
    elif strategy == "index":
        if k == 1:
            indices = [frame_count // 2]  # Middle frame
        elif k == 2:
            indices = [0, frame_count - 1]  # First and last
        elif k == 3:
            indices = [0, frame_count // 2, frame_count - 1]
        else:
            indices = [int(frame_count * (i + 1) / (k + 1)) for i in range(k)]
    elif strategy == "random":
        indices = sorted(random.sample(range(frame_count), k))
    else:
        raise ValueError(f"Unknown sampling strategy: {strategy}")
    
    # Extract frames
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame_data = cap.read()
        if not ret:
            continue
        frame_data = cv2.cvtColor(frame_data, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_data))
    
    cap.release()
    return frames

def make_conversation(row):
    """
    Create conversation format for RL training - matches supervised finetuning format.
    Key difference: Multiple images (25 frames) in the content list.
    """
    video_name = str(row["video_name"]).strip()
    video_path = VIDEO_ROOT / video_name
    
    # Sample 25 frames from the video
    frames = sample_frames_from_video(video_path)
    
    # Build content list: text prompt first, then all 25 images
    content = [{"type": "text", "text": PROMPT_TEMPLATE}]
    content += [{"type": "image", "image": img} for img in frames]
    
    # Construct the prompt in the conversation format
    prompt = [
        {
            "role": "user",
            "content": content,
        },
    ]
    
    # Ground truth answer (for reward calculation)
    gt_json = strip_fences(row["ground_truth_raw"])
    
    return {
        "prompt": prompt,
        "answer": gt_json,
        "video_name": video_name,
    }

print("Video sampling and conversation functions defined")
print(f"Each video will be represented by {NUM_FRAMES_PER_VIDEO} frames")
print("This matches the supervised finetuning setup")

In [ ]:
# Convert to dataset format with multiprocessing
from datasets import Dataset
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import functools

print("Converting train dataset to conversation format...")
print(f"This will take a few minutes as we sample 25 frames from each {len(train_df)} videos...")

# Define a wrapper function that can be pickled for multiprocessing
def process_row_wrapper(args):
    """Wrapper to process a single row - needed for multiprocessing"""
    idx, row_dict = args
    try:
        # Convert row_dict back to a pandas Series-like object
        class RowLike:
            def __init__(self, d):
                for k, v in d.items():
                    setattr(self, k, v)
            def __getitem__(self, key):
                return getattr(self, key)
        
        row = RowLike(row_dict)
        return make_conversation(row)
    except Exception as e:
        print(f"Error processing video {row_dict.get('video_name', 'unknown')}: {e}")
        return None

# Prepare data for multiprocessing - convert DataFrame rows to dicts
row_data = [(idx, row.to_dict()) for idx, row in train_df.iterrows()]

# Use multiprocessing for faster processing
num_workers = min(cpu_count(), 8)  # Use up to 8 workers
print(f"Using {num_workers} worker processes for parallel video loading...\n")

with Pool(num_workers) as pool:
    train_data = list(tqdm(
        pool.imap(process_row_wrapper, row_data),
        total=len(row_data),
        desc="Processing videos"
    ))

# Filter out any None results from errors
train_data = [d for d in train_data if d is not None]

train_dataset = Dataset.from_list(train_data)

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Sample keys: {list(train_dataset[0].keys())}")
print(f"\nNote: Each sample contains a 'prompt' with 25 images + text, matching supervised finetuning")
print(f"Processing time significantly reduced using {num_workers} parallel workers!")

### 5) Apply Chat Template

In [ ]:
train_dataset = train_dataset.map(
    lambda example: {
        "prompt": tokenizer.apply_chat_template(
            example["prompt"],
            tokenize = False,
            add_generation_prompt = True,  # Must add assistant
        )
    }
)

print("Sample prompt after chat template:")
print(train_dataset[0]["prompt"][:500] + "...")

## Reward Functions

We define two reward functions:
1. **Formatting reward**: Checks for proper `<REASONING>` and `<SOLUTION>` structure
2. **Correctness reward**: Smooth scoring based on count accuracy

In [ ]:
def _extract_solution_block(text: str) -> str | None:
    """Extract content between <SOLUTION> tags"""
    if not isinstance(text, str):
        return None
    m = re.search(f"{re.escape(SOLUTION_START)}(.*?){re.escape(SOLUTION_END)}", text, flags=re.DOTALL)
    return m.group(1).strip() if m else None

def _parse_counts_from_text(text: str) -> Dict[str, int] | None:
    """Parse object counts from solution text"""
    if not isinstance(text, str):
        return None
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.DOTALL).strip()
    
    # Try JSON parsing
    try:
        obj = json.loads(s)
        inner = obj.get("object_counts", obj) if isinstance(obj, dict) else None
        if isinstance(inner, dict):
            out = {}
            for k in REQUIRED_KEYS:
                v = inner.get(k, None)
                if isinstance(v, (int, float)):
                    out[k] = int(round(v))
                else:
                    return None
            return out
    except Exception:
        pass
    
    # Fallback: extract 4 integers in order
    nums = [int(x) for x in re.findall(r"[-+]?\d+", s)]
    if len(nums) >= 4:
        return {k: max(0, int(nums[i])) for i, k in enumerate(REQUIRED_KEYS)}
    return None

def formatting_reward_func(completions: List[str], **kwargs) -> List[float]:
    """Reward for proper formatting"""
    scores = []
    for c in completions:
        score = 0.0
        if isinstance(c, str):
            # Check for REASONING block
            think = re.findall(f"{re.escape(REASONING_START)}(.*?){re.escape(REASONING_END)}", c, flags=re.DOTALL)
            if len(think) == 1:
                score += 0.5
            
            # Check for SOLUTION block
            ans = re.findall(f"{re.escape(SOLUTION_START)}(.*?){re.escape(SOLUTION_END)}", c, flags=re.DOTALL)
            if len(ans) == 1:
                score += 0.5
                # Bonus if JSON parses correctly
                if _parse_counts_from_text(ans[0]) is not None:
                    score += 0.5
            
            # Penalty for spam (addCriterion issue)
            removal = c.replace("addCriterion", "").replace("\n", "")
            if len(c) and (len(c) - len(removal)) / len(c) >= 0.5:
                score -= 1.0
            
            # Penalty for overly verbose solutions
            if ans and len(ans[0]) > 1200:
                score -= 0.5
        
        scores.append(float(score))
    return scores

def _per_key_score(gt: int, pred: int, cap: int = 3) -> float:
    """Smooth scoring for individual count accuracy"""
    d = abs(int(gt) - int(pred))
    d = min(d, cap)
    return 1.0 / (1.0 + d)

def correctness_reward_func(prompts: List[str], completions: List[str], answer: List[str], **kwargs) -> List[float]:
    """Reward for correct object counts"""
    # Parse ground truths
    gts = []
    for a in answer:
        try:
            s = strip_fences(a)
            obj = json.loads(s)
            inner = obj.get("object_counts", obj)
            gts.append({k: int(inner.get(k, 0)) for k in REQUIRED_KEYS})
        except Exception:
            gts.append({k: 0 for k in REQUIRED_KEYS})
    
    # Score predictions
    out = []
    for comp, gt in zip(completions, gts):
        sol = _extract_solution_block(comp if isinstance(comp, str) else "")
        pred = _parse_counts_from_text(sol or "")
        
        if pred is None:
            out.append(0.0)
            continue
        
        # Compute per-key scores
        per_key = [_per_key_score(gt[k], pred.get(k, 0)) for k in REQUIRED_KEYS]
        s = 0.5 + 1.5 * (sum(per_key) / len(per_key))  # Base + smooth correctness
        
        # Logical consistency check: driver count should not exceed forklift count
        if pred.get("forklift_driver_count", 0) > pred.get("forklift_count", 0):
            s -= 0.25
        
        out.append(float(s))
    
    # Debug: print first example
    if prompts:
        print('-' * 20, f"Sample Answer: {answer[0][:200]}")
        print(f"Sample Completion: {completions[0][:300]}...")
    
    return out

## Test Reward Functions

Quick sanity check on reward functions

In [ ]:
example_answer = strip_fences(train_df.iloc[0]["ground_truth_raw"])
answers = [example_answer, example_answer, example_answer]

good_pred = (
    f"{REASONING_START}I tracked each unique object across all frames to maintain temporal consistency.{REASONING_END}"
    f"{SOLUTION_START}"
    + example_answer +
    f"{SOLUTION_END}"
)

off_by_one = f"{REASONING_START}Counted objects in video{REASONING_END}{SOLUTION_START}{{\"object_counts\":{{\"forklift_count\":1,\"pedestrian_count\":0,\"forklift_driver_count\":0,\"truck_count\":0}}}}{SOLUTION_END}"
bad_output = "There might be some objects in the video."

completions = [good_pred, off_by_one, bad_output]
fmt_scores = formatting_reward_func(completions)
corr_scores = correctness_reward_func(["prompt"] * 3, completions, answers)

print("Formatting scores:", fmt_scores)
print("Correctness scores:", corr_scores)
print("\nExpected: good_pred > off_by_one > bad_output")

## Pre-training Inference

Let's test the model before RL training

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)

test_idx = 10  # Try sample 10

# Extract images from the prompt content for multi_modal_data
# The prompt contains: [{"role": "user", "content": [{"type": "text"}, {"type": "image", "image": img1}, ...]}]
user_content = train_dataset[test_idx]["prompt"]
if isinstance(user_content, str):
    # Already processed through chat template
    prompt_text = user_content
    # For images, we need to get them from the original data before chat template
    # Since we can't easily extract them after tokenization, let's reconstruct
    # For now, we'll note this limitation
    print("Note: Using tokenized prompt. Images are embedded in the conversation.")
    outputs = model.fast_generate(
        {"prompt": prompt_text},
        sampling_params,
    )
else:
    # Before chat template - has the full conversation structure
    prompt_text = tokenizer.apply_chat_template(
        user_content,
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = model.fast_generate(
        {"prompt": prompt_text},
        sampling_params,
    )

print("\n=== Before RL Training ===")
print(f"Video: {train_dataset[test_idx]['video_name']}")
print(f"Ground Truth: {train_dataset[test_idx]['answer'][:200]}")
print(f"\nModel Output:\n{outputs[0].outputs[0].text}")

## Training Configuration (GRPO)

Set up the GRPO trainer with GSPO enabled

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,  # Increase to 4 for smoother training
    num_generations = 4,  # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 0.5,  # Set to 1 for full training run
    # max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none",  # Can use Weights & Biases
    output_dir = str(OUT_DIR),

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

## Train the Model

**Important**: You might see 0 reward for the first 100-150 steps. Be patient!

You may also see `addCriterion` or weird outputs initially - this is a known Qwen2.5-VL quirk that improves with training.

In [ ]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train()

## Post-training Inference

Test the RL-trained model

In [ ]:
# Save LoRA first
model.save_lora("grpo_object_counting_lora")

In [ ]:
# Test with RL-trained model
test_idx = 10

# Get the prompt (already chat-templated)
prompt_text = train_dataset[test_idx]["prompt"]

outputs = model.fast_generate(
    {"prompt": prompt_text},
    sampling_params,
    lora_request = model.load_lora("grpo_object_counting_lora")
)

print("\n=== After RL Training ===")
print(f"Video: {train_dataset[test_idx]['video_name']}")
print(f"Ground Truth: {train_dataset[test_idx]['answer'][:200]}")
print(f"\nModel Output:\n{outputs[0].outputs[0].text}")

## Save Model

Save the trained model in various formats

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if True:
    model.save_pretrained("grpo_object_counting_final")
    tokenizer.save_pretrained("grpo_object_counting_final")
if False:
    model.push_to_hub("hf/grpo-object-counting", token = "")
    tokenizer.push_to_hub("hf/grpo-object-counting", token = "")

## Evaluation on Test Set

Evaluate the trained model on the held-out test set

In [ ]:
# Prepare test dataset with multiprocessing
from multiprocessing import Pool, cpu_count

print("Preparing test dataset...")
print(f"Sampling 25 frames from each of {len(test_df)} test videos...")

# Prepare data for multiprocessing
test_row_data = [(idx, row.to_dict()) for idx, row in test_df.iterrows()]

# Use multiprocessing for faster processing
num_workers = min(cpu_count(), 8)
print(f"Using {num_workers} worker processes...\n")

with Pool(num_workers) as pool:
    test_data = list(tqdm(
        pool.imap(process_row_wrapper, test_row_data),
        total=len(test_row_data),
        desc="Processing test videos"
    ))

# Filter out any None results from errors
test_data = [d for d in test_data if d is not None]

test_dataset = Dataset.from_list(test_data)

test_dataset = test_dataset.map(
    lambda example: {
        "prompt": tokenizer.apply_chat_template(
            example["prompt"],
            tokenize = False,
            add_generation_prompt = True,
        )
    }
)

print(f"\nTest dataset size: {len(test_dataset)}")
print(f"Each test sample has 25 frames from the video")

In [ ]:
# Run inference on test set (sample of first 10 for quick check)
import numpy as np

sampling_params_eval = SamplingParams(
    temperature = 0.2,  # Lower temperature for more consistent evaluation
    top_k = 50,
    max_tokens = 512,
)

eval_sample_size = min(10, len(test_dataset))
mae_per_key = {k: [] for k in REQUIRED_KEYS}

print(f"Evaluating on {eval_sample_size} test samples...\n")

for i in tqdm(range(eval_sample_size), desc="Evaluating"):
    prompt_text = test_dataset[i]["prompt"]
    
    outputs = model.fast_generate(
        {"prompt": prompt_text},
        sampling_params_eval,
        lora_request = model.load_lora("grpo_object_counting_lora")
    )
    
    # Parse prediction
    completion = outputs[0].outputs[0].text
    sol = _extract_solution_block(completion)
    pred = _parse_counts_from_text(sol or "")
    
    # Parse ground truth
    gt_json = json.loads(strip_fences(test_dataset[i]["answer"]))
    gt = gt_json.get("object_counts", {})
    
    if pred:
        for k in REQUIRED_KEYS:
            mae_per_key[k].append(abs(gt.get(k, 0) - pred.get(k, 0)))
    
    print(f"\nSample {i+1}: {test_dataset[i]['video_name']}")
    print(f"GT: {gt}")
    print(f"Pred: {pred}")

# Compute average MAE
print("\n" + "="*50)
print(f"Average MAE per object type (on {eval_sample_size} samples):")
for k in REQUIRED_KEYS:
    if mae_per_key[k]:
        avg_mae = np.mean(mae_per_key[k])
        print(f"  {k}: {avg_mae:.2f}")

---

## Done!

You've successfully trained a Qwen2.5-VL-7B model with GRPO for object counting with temporal consistency.

### Next Steps:
1. Run full evaluation on the complete test set
2. Analyze temporal consistency improvements
3. Fine-tune hyperparameters for better performance
4. Export model for deployment

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>